# Vision-Transformer CRP — Walkthrough

Visual end-to-end exploration of the four ViT concept-detector classes added in this fork. The notebook is **visualisation-first**: every section produces a comparative grid of heatmaps, with numeric relevance shown as subplot labels rather than printed tables.

## What you'll see

1. **Setup + dataset** — pick `DATASET_NAME`, model, and target image.
2. **Concept atlas** — for one target image, a single grid covering all four concept granularities × multiple blocks × top-K most-relevant concepts per cell.
3. **Per-head breakdown** — every head's `HeadConcept` heatmap on the same image, sorted by relevance.
4. **Layer evolution** — one head's `HeadConcept` traced through early / mid / late blocks of the network.
5. **K vs Q vs V** — three heatmaps for one head, splitting the attention input into its query / key / value contributions.
6. **Head-dim closeup** — the top-N `HeadDimConcept`s within a single head, exposing per-dimension fine structure.
7. **Reference samples** — for each granularity, the top-K dataset images that maximise each top concept's relevance, with each reference image's conditional heatmap overlaid (FV-indexed).

**Theory**: AttnLRP (Achtibat et al., ICML 2024; [arXiv 2402.05602](https://arxiv.org/abs/2402.05602)) on top of CRP (Achtibat et al., Nature MI 2023; [arXiv 2206.03208](https://arxiv.org/abs/2206.03208)).

**Concept cheat sheet** (two orthogonal granularity axes — *split by K/Q/V?* and *split by head_dim?*):

| Class | Tap | Granularity | `attribute()` shape |
|---|---|---|---|
| `HeadConcept`        | `attn_out_tap` | per head (output tokens)              | `(B, num_heads)`              |
| `HeadDimConcept`     | `attn_out_tap` | per `(head, dim)` (output tokens)     | `(B, num_heads, head_dim)`    |
| `KQVHeadConcept`     | `qkv_tap`      | per `(part, head)` (K/Q/V projections) | `(B, 3, num_heads)`         |
| `KQVHeadDimConcept`  | `qkv_tap`      | per `(part, head, dim)` (K/Q/V projections) | `(B, 3, num_heads, head_dim)` |

## 1. Setup

Imports and path bootstrap. `experiments/` lives on `sys.path` so the shared `viz.py` plotting helpers and `datasets.py` loader resolve.

In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
import timm
from timm.data import resolve_data_config, create_transform

from crp.attribution import CondAttribution
from crp.transformer_patches import (
    AttnLRPEpsilonComposite, AttnLRPGammaComposite,
)

# Locate the repo root so the notebook works whether launched from
# the repo root or from tutorials/vit_crp/. All artefacts under <repo>/data/.
def _repo_root():
    p = Path.cwd().resolve()
    while p != p.parent:
        if (p / 'pyproject.toml').is_file():
            return p
        p = p.parent
    raise RuntimeError('repo root with pyproject.toml not found above CWD')
REPO_ROOT = _repo_root()
DATA_DIR = REPO_ROOT / 'data'
FV_ROOT = DATA_DIR / 'feature_visualization'
DATA_DIR.mkdir(parents=True, exist_ok=True)
FV_ROOT.mkdir(parents=True, exist_ok=True)

sys.path.insert(0, str(REPO_ROOT / 'experiments'))
from datasets import load as load_dataset, IMAGENETTE_CLASS_NAMES  # noqa: E402
from viz import (  # noqa: E402
    CONCEPT_CLASSES, denormalize,
    plot_concept_atlas, plot_per_head, plot_layer_evolution,
    plot_kqv_split, plot_head_dim_grid, plot_reference_samples,
)

torch.set_grad_enabled(True)
print('torch', torch.__version__, '| timm', timm.__version__)

## 2. Configuration

All knobs in one place. Override here for a quicker run on a smaller model, a different dataset, or a different image.

* `MODEL_NAME` — `vit_base_patch16_224` (86 M, 12 blocks, 12 heads, head_dim 64) is the default and the size at which most of the visualisation work was developed. `vit_small_patch16_224` (22 M, 6 heads) and `vit_tiny_patch16_224` (5 M, 3 heads) run faster on CPU.
* `DATASET_NAME` — `'imagenette'` (10-class, 98 MB, auto-downloaded) is the default. Switch to `'imagenet_val'` once `data/imagenet_val/` is populated; see `experiments/datasets.py`.
* `BLOCKS_OF_INTEREST` — which attention blocks to include in the multi-block atlas. Mid-network blocks (5–9 on a 12-block ViT) tend to carry the most class-relevant structure.
* `TARGET_INDEX` — which dataset item to attribute. `None` picks one deterministically from `RANDOM_SEED`.
* `USE_GAMMA` / `GAMMA` — γ-LRP (the AttnLRP §3.2.1 default) is available but **not** the default here. On `vit_base` the γ-LRP rule stack inflates relevance by ~10¹⁶× (see [`CURRENT_STATE.md`](../../CURRENT_STATE.md) Milestone D), making heatmaps numerically degenerate — every pixel ends up at the colour extremes. Plain ε-LRP gives clean, interpretable heatmaps and is the default. Toggle on at your own risk.

In [ ]:
MODEL_NAME = 'vit_base_patch16_224'   # 'vit_small_patch16_224' / 'vit_tiny_patch16_224'
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

DATASET_NAME = 'imagenette'           # 'imagenette' | 'imagenet_val'
NUM_SAMPLES = 64                      # for the FV index — bump up for nicer reference samples

BLOCKS_OF_INTEREST = [3, 6, 9, 11]    # 4 blocks for the atlas (rows × cols)
MID_BLOCK = 6                         # block used for per-head / kqv_split / head_dim sections
TOP_K = 4                             # top concepts per granularity

TARGET_INDEX = None                   # int → that index; None → random under RANDOM_SEED
RANDOM_SEED = 0

USE_GAMMA = False                     # γ-LRP catastrophically inflates on vit_base; see Milestone D
GAMMA = 0.25
EPSILON = 1e-6

print(f'device  : {DEVICE}')
print(f'model   : {MODEL_NAME}')
print(f'dataset : {DATASET_NAME}')
print(f'blocks  : {BLOCKS_OF_INTEREST}  (mid={MID_BLOCK})')
print(f'rule    : {("γ-LRP, γ=" + str(GAMMA)) if USE_GAMMA else "ε-LRP"}')

## 3. Dataset, model, and composite

Three short cells. `load_dataset(...)` auto-downloads Imagenette on first call (~98 MB). The composite bundles `TimmViTCanonizer`, which installs the `qkv_tap` and `attn_out_tap` `nn.Identity` submodules and swaps `forward` on the timm Attention to embed AttnLRP's autograd rules — all scoped to `composite.context()` and reverted on exit.

In [ ]:
# Sample evenly across classes for a balanced FV index.
if DATASET_NAME == 'imagenette':
    n_per_class = max(1, NUM_SAMPLES // 10)
    classes = None
elif DATASET_NAME == 'imagenet_val':
    n_per_class = max(1, NUM_SAMPLES // 1000)
    classes = None
else:
    raise ValueError(DATASET_NAME)

# Step 1: model (load before the dataset so we can resolve preprocessing).
model = timm.create_model(MODEL_NAME, pretrained=True).eval().to(DEVICE)
cfg = resolve_data_config({}, model=model)
preprocess_fn = create_transform(**cfg)

# Step 2: dataset (with the timm preprocess pipeline).
dataset = load_dataset(
    DATASET_NAME, root=DATA_DIR, n_per_class=n_per_class,
    classes=classes, seed=RANDOM_SEED,
    transform=preprocess_fn,
)
print(f'{dataset.name}: {len(dataset)} images, {dataset.num_classes} classes')

# Step 3: composite + attribution.
if USE_GAMMA:
    composite = AttnLRPGammaComposite(gamma=GAMMA, epsilon=EPSILON)
else:
    composite = AttnLRPEpsilonComposite(epsilon=EPSILON)
attribution = CondAttribution(model, device=torch.device(DEVICE))

NUM_HEADS = model.blocks[MID_BLOCK].attn.num_heads
HEAD_DIM = model.blocks[MID_BLOCK].attn.head_dim
print(f'{type(composite).__name__}: num_heads={NUM_HEADS}, head_dim={HEAD_DIM}')

## 4. Pick a target image

We attribute under the image's *true* class (so the relevance lines up with what's actually in the picture). Re-run this cell with a different `RANDOM_SEED` to pick another image.

In [ ]:
rng = np.random.default_rng(RANDOM_SEED)
if TARGET_INDEX is None:
    target_idx = int(rng.integers(0, len(dataset)))
else:
    target_idx = TARGET_INDEX
target_data, target_class = dataset[target_idx]
target_pre = target_data.unsqueeze(0).to(DEVICE).requires_grad_(True)
class_name = IMAGENETTE_CLASS_NAMES.get(target_class, str(target_class))

fig, ax = plt.subplots(1, 1, figsize=(3, 3))
ax.imshow(denormalize(target_pre, model))
ax.set_xticks([]); ax.set_yticks([])
ax.set_title(f'idx {target_idx}  •  cls {target_class}  •  {class_name}', fontsize=10)
plt.show()

## 5. Concept atlas — four granularities × multiple blocks

Each row is one `(granularity, block)` pair; columns are the top-`TOP_K` concepts of that granularity at that block, ranked by absolute relevance under the target class. Each cell shows the conditional heatmap overlaid on the input; subplot titles report the concept id and its raw relevance score.

Read top to bottom: each granularity's view of the same image at different network depths.

In [ ]:
fig = plot_concept_atlas(
    target_pre, model, attribution, composite,
    blocks=BLOCKS_OF_INTEREST, target_class=target_class, top_k=TOP_K,
    suptitle=(f'concept atlas  •  {class_name}  •  {MODEL_NAME}  •  '
              f'{type(composite).__name__}'),
)
plt.show()

## 6. Per-head breakdown — every head's HeadConcept on the same image

At one mid-network block, render `HeadConcept`'s heatmap for every head, sorted by descending |relevance|. Each head attends to different spatial structures of the same target — strongly relevant heads highlight the salient object, weakly relevant heads tend to wander or track context.

In [ ]:
fig = plot_per_head(
    target_pre, model, attribution, composite,
    block_idx=MID_BLOCK, target_class=target_class,
    suptitle=f'HeadConcept on every head  •  block {MID_BLOCK}  •  {class_name}',
)
plt.show()

## 7. Layer evolution — one head across blocks

Pick the head with the highest absolute relevance at the mid block, then trace its `HeadConcept` heatmap through early / mid / late blocks. Demonstrates how the same conceptual head shifts spatial focus through the network's depth.

In [ ]:
# Pick top-1 head at the mid block.
from viz import _per_concept_scores, _enumerate_ids  # internal but useful here
concept_h = CONCEPT_CLASSES['head'](model)
layer_mid = f'blocks.{MID_BLOCK}.attn.{concept_h.tap_name}'
scores_mid = _per_concept_scores(
    attribution, composite, target_pre, layer_mid, concept_h, target_class,
)
best_head = int(torch.argmax(scores_mid.abs()))
print(f'tracing head {best_head}  (mid-block relevance = {scores_mid[best_head].item():+.3f})')

fig = plot_layer_evolution(
    target_pre, model, attribution, composite,
    concept_class=CONCEPT_CLASSES['head'], concept_id=best_head,
    blocks=tuple(range(0, len(model.blocks), max(1, len(model.blocks) // 6))),
    target_class=target_class,
    suptitle=f'HeadConcept(h{best_head})  •  evolution across blocks  •  {class_name}',
)
plt.show()

## 8. K vs Q vs V — splitting the same head

For the same `best_head` at the mid block, render its three K/Q/V components separately via `KQVHeadConcept`. Each part can highlight a different aspect of the same target (often K is *where to look*, V is *what value to return*).

In [ ]:
fig = plot_kqv_split(
    target_pre, model, attribution, composite,
    block_idx=MID_BLOCK, head_id=best_head, target_class=target_class,
    suptitle=f'KQVHeadConcept  •  block {MID_BLOCK}, h{best_head}  •  {class_name}',
)
plt.show()

## 9. Head-dim closeup — fine structure inside one head

Within `best_head`, rank the `head_dim` channels by absolute relevance and render the top-N. Demonstrates the per-dim granularity of `HeadDimConcept` — most heads have a few dominant dims that carry the class-relevant signal.

In [ ]:
fig = plot_head_dim_grid(
    target_pre, model, attribution, composite,
    block_idx=MID_BLOCK, head_id=best_head, target_class=target_class,
    n_dims=8,
    suptitle=f'HeadDimConcept top-8  •  block {MID_BLOCK}, h{best_head}  •  {class_name}',
)
plt.show()

## 10. Build a FeatureVisualization index per granularity

For each of the four concept classes we build a separate FV index — same model, different concept layer/aggregation, different number of concepts. Each index ranks dataset samples by per-concept relevance under the sample's true class.

FV writes its results to `data/feature_visualization/<name>/` and the cell skips `fv.run()` for any granularity that already has an index there. **Slow on first run** (~30 s/granularity for `NUM_SAMPLES=64` on `vit_base`); subsequent calls are cache hits. Delete the directory to force a rebuild.

In [ ]:
from crp.visualization import FeatureVisualization

concepts: dict = {}
fvs: dict = {}
layers: dict = {}
for name, cls in CONCEPT_CLASSES.items():
    concept = cls(model)
    concepts[name] = concept
    layer_name = f'blocks.{MID_BLOCK}.attn.{concept.tap_name}'
    layers[name] = layer_name
    fvs[name] = FeatureVisualization(
        attribution, dataset, layer_map={layer_name: concept},
        preprocess_fn=preprocess_fn, path=str(FV_ROOT / name),
        device=torch.device(DEVICE),
    )

for name, layer in layers.items():
    print(f'  {name:13} -> {layer}')

In [ ]:
%%time
for name, fv in fvs.items():
    rel_dir = FV_ROOT / name / 'RelMax_sum_normed'
    if rel_dir.is_dir() and any(rel_dir.glob('*.npy')):
        print(f'[{name}] cached index found at {rel_dir} — skipping')
        continue
    print(f'\n=== running FV index for {name!r} ===')
    fv.run(composite, 0, len(dataset), batch_size=8, checkpoint=10000)
print('\nall four indices ready.')

## 11. Reference samples — concept representatives across the dataset

For each granularity, pick the top-`TOP_K` concepts on the target image, then for each concept fetch the top-`TOP_K` *reference samples* — dataset images that maximise that concept's relevance — and render each with the concept's conditional heatmap overlaid. The right-most column shows the same concept on the *target* image, for direct comparison.

These representatives are the concept's visual signature: across many different scenes, what spatial pattern does this concept consistently fire on?

In [ ]:
fig = plot_reference_samples(
    fvs['head'], dataset,
    concept_name='head', block_idx=MID_BLOCK,
    target_image=target_pre, target_class=target_class,
    model=model, attribution=attribution, composite=composite,
    n_top_concepts=TOP_K, n_refs_per_concept=4,
    suptitle=f'head top-{TOP_K} concepts  •  block {MID_BLOCK}  •  references + target',
)
plt.show()

In [ ]:
fig = plot_reference_samples(
    fvs['head_dim'], dataset,
    concept_name='head_dim', block_idx=MID_BLOCK,
    target_image=target_pre, target_class=target_class,
    model=model, attribution=attribution, composite=composite,
    n_top_concepts=TOP_K, n_refs_per_concept=4,
    suptitle=f'head_dim top-{TOP_K} concepts  •  block {MID_BLOCK}  •  references + target',
)
plt.show()

In [ ]:
fig = plot_reference_samples(
    fvs['kqv_head'], dataset,
    concept_name='kqv_head', block_idx=MID_BLOCK,
    target_image=target_pre, target_class=target_class,
    model=model, attribution=attribution, composite=composite,
    n_top_concepts=TOP_K, n_refs_per_concept=4,
    suptitle=f'kqv_head top-{TOP_K} concepts  •  block {MID_BLOCK}  •  references + target',
)
plt.show()

In [ ]:
fig = plot_reference_samples(
    fvs['kqv_head_dim'], dataset,
    concept_name='kqv_head_dim', block_idx=MID_BLOCK,
    target_image=target_pre, target_class=target_class,
    model=model, attribution=attribution, composite=composite,
    n_top_concepts=TOP_K, n_refs_per_concept=4,
    suptitle=f'kqv_head_dim top-{TOP_K} concepts  •  block {MID_BLOCK}  •  references + target',
)
plt.show()

## 12. Notes

* The four sections above (atlas, per-head, layer evolution, K/Q/V split, head_dim closeup, reference samples) are systematic: every concept variant gets a visual treatment, every dimension of granularity is shown across at least one axis. Numerical relevance scores live as subplot labels.
* For the cleanest heatmaps on `vit_base`, leave `USE_GAMMA = True` (γ-LRP, AttnLRP §3.2.1). Toggle off to compare against ε-LRP.
* Bump `NUM_SAMPLES` to 128 / 256 / … for nicer reference samples (FV indexes more dataset images and the top-N representatives become more consistent). The atlas / per-head / layer / K/Q/V / head_dim sections don't depend on FV and run in <1 minute regardless.
* Quantitative benchmark numbers (Petsiuk deletion / insertion AUC, PA-LRP, residual-LRP) live in [`experiments.ipynb`](experiments.ipynb) and the milestone CSVs under `data/`. This notebook is the **visual** side of the same investigation.
* See [`CURRENT_STATE.md`](../../CURRENT_STATE.md) for the design and the milestone-A/D/G findings, and [`FUTURE_STATE.md`](../../FUTURE_STATE.md) for the open follow-ups.